# Optimization with Differentiable Flowsheets

This notebook demonstrates various optimization scenarios enabled by automatic differentiation.

## Optimization Scenarios

1. **Single-variable optimization** - Find optimal temperature
2. **Multi-variable optimization** - Jointly optimize V and T
3. **Constrained optimization** - Meet conversion targets
4. **Economic optimization** - Maximize profit
5. **Parameter estimation** - Fit kinetics to data
6. **Pareto analysis** - Multi-objective trade-offs

All optimizations use **gradient-based methods** enabled by JAX automatic differentiation.

In [ ]:
import jax
import jax.numpy as jnp
from jax import Array
from typing import Callable

jax.config.update("jax_enable_x64", True)

from difflow.streams import Stream, make_stream, get_flows
from difflow.thermo import IdealThermo, SpeciesData
from difflow.units.cstr import CSTR, CSTRParams

## Setup

In [ ]:
species_data = {
    "A": SpeciesData(
        name="A", MW=100.0, Cp_coeffs=(75.0, 0.0, 0.0, 0.0),
        Hvap_coeffs=(35000.0, 0.38, 500.0), antoine_coeffs=(10.0, 3000.0, -50.0),
    ),
    "B": SpeciesData(
        name="B", MW=100.0, Cp_coeffs=(75.0, 0.0, 0.0, 0.0),
        Hvap_coeffs=(30000.0, 0.38, 450.0), antoine_coeffs=(10.0, 2800.0, -40.0),
        Hf=-50000.0,
    ),
}

thermo = IdealThermo(species_data)
species_order = ["A", "B"]
stoichiometry = jnp.array([[-1.0], [+1.0]])


def rate_function(C: dict[str, Array], T: Array, params: dict) -> Array:
    k = params["A"] * jnp.exp(-params["Ea"] / (8.314 * T))
    return jnp.array([k * C["A"]])


def create_cstr(V: Array, rate_params: dict) -> CSTR:
    params = CSTRParams(
        V=V,
        rate_fn=rate_function,
        stoich=stoichiometry,
        rate_params=rate_params,
        species_order=species_order,
        dH_rxn=jnp.array([-50000.0]),
    )
    return CSTR(params, thermo=thermo, mode="isothermal")

print("Setup complete ✓")

## Optimizer Utilities

We'll implement two common optimizers:
1. **Gradient Descent** - Simple but effective
2. **Adam** - Adaptive learning rates for better convergence

In [ ]:
def gradient_descent(
    objective: Callable,
    x0: Array,
    learning_rate: float | Array = 0.01,
    max_iter: int = 100,
    bounds: tuple | None = None,
    verbose: bool = True,
) -> tuple[Array, list]:
    """Simple gradient descent optimizer."""
    x = x0
    lr = jnp.asarray(learning_rate)
    history = []
    
    for i in range(max_iter):
        obj = objective(x)
        grad = jax.grad(objective)(x)
        grad_norm = jnp.linalg.norm(grad)
        
        history.append((x.copy(), float(obj), float(grad_norm)))
        x = x - lr * grad
        
        if bounds is not None:
            lower, upper = bounds
            x = jnp.clip(x, lower, upper)
        
        if verbose and (i + 1) % 10 == 0:
            print(f"  Iter {i+1:3d}: obj = {float(obj):.6f}, |grad| = {float(grad_norm):.6f}")
    
    return x, history


def adam_optimizer(
    objective: Callable,
    x0: Array,
    learning_rate: float = 0.01,
    max_iter: int = 100,
    beta1: float = 0.9,
    beta2: float = 0.999,
    bounds: tuple | None = None,
    verbose: bool = True,
) -> tuple[Array, list]:
    """Adam optimizer for better convergence."""
    x = x0
    m = jnp.zeros_like(x)  # First moment
    v = jnp.zeros_like(x)  # Second moment
    history = []
    eps = 1e-8
    
    for i in range(max_iter):
        obj = objective(x)
        grad = jax.grad(objective)(x)
        
        history.append((x.copy(), float(obj), float(jnp.linalg.norm(grad))))
        
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * grad ** 2
        
        m_hat = m / (1 - beta1 ** (i + 1))
        v_hat = v / (1 - beta2 ** (i + 1))
        
        x = x - learning_rate * m_hat / (jnp.sqrt(v_hat) + eps)
        
        if bounds is not None:
            x = jnp.clip(x, bounds[0], bounds[1])
        
        if verbose and (i + 1) % 20 == 0:
            print(f"  Iter {i+1:3d}: obj = {float(obj):.6f}")
    
    return x, history

print("Optimizers defined ✓")

## 1. Single-Variable Optimization: Optimal Temperature

**Goal**: Find the temperature that maximizes conversion

This is a simple 1D optimization problem that demonstrates gradient-based search.

In [ ]:
def neg_conversion(T: Array) -> Array:
    """Negative conversion (to minimize)."""
    cstr = create_cstr(
        V=jnp.array(1.0),
        rate_params={"A": jnp.array(1e6), "Ea": jnp.array(50000.0)},
    )
    inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
    _, info = cstr(inlet, T_spec=T)
    return -info["conversion"]["A"]


print("Finding T that maximizes conversion...\n")

T_opt, history = gradient_descent(
    neg_conversion,
    x0=jnp.array(350.0),
    learning_rate=5.0,
    max_iter=50,
    bounds=(jnp.array(300.0), jnp.array(500.0)),
    verbose=False,
)

print(f"✓ Optimal temperature: T* = {float(T_opt):.1f} K")
print(f"  Maximum conversion: {-float(neg_conversion(T_opt))*100:.2f}%")

print("\nConversion vs Temperature:")
for T in [300, 350, 400, 450, 500]:
    X = -neg_conversion(jnp.array(float(T)))
    bar = "█" * int(float(X) * 50)
    print(f"  T = {T} K: {float(X)*100:5.1f}% {bar}")

## 2. Multi-Variable Optimization: V and T Jointly

**Goal**: Minimize cost while achieving target conversion

$$\min_{V, T} \quad \text{Capital}(V) + \text{Energy}(T)$$
$$\text{s.t.} \quad X \geq 95\%$$

We use a penalty method to handle the constraint.

In [ ]:
def objective_with_penalty(params: Array) -> Array:
    """Minimize cost subject to conversion >= 95%."""
    V, T = params[0], params[1]
    
    cstr = create_cstr(
        V=V,
        rate_params={"A": jnp.array(1e6), "Ea": jnp.array(50000.0)},
    )
    inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
    _, info = cstr(inlet, T_spec=T)
    
    conversion = info["conversion"]["A"]
    
    # Cost function
    capital_cost = 10000.0 * V  # $10,000 per m³
    energy_cost = 100.0 * (T - 300.0)  # $100 per K above 300
    total_cost = capital_cost + energy_cost
    
    # Penalty for missing conversion target
    target = 0.95
    penalty = 1e6 * jnp.maximum(0.0, target - conversion) ** 2
    
    return total_cost + penalty


print("Minimizing: Capital + Energy cost")
print("Subject to: Conversion >= 95%\n")

x_opt, history = adam_optimizer(
    objective_with_penalty,
    x0=jnp.array([1.0, 400.0]),
    learning_rate=0.05,
    max_iter=200,
    bounds=(jnp.array([0.1, 300.0]), jnp.array([10.0, 500.0])),
    verbose=False,
)

V_opt, T_opt = float(x_opt[0]), float(x_opt[1])

# Verify solution
cstr = create_cstr(jnp.array(V_opt), {"A": jnp.array(1e6), "Ea": jnp.array(50000.0)})
inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
_, info = cstr(inlet, T_spec=jnp.array(T_opt))

print("✓ Optimal design:")
print(f"  Volume V* = {V_opt:.3f} m³")
print(f"  Temperature T* = {T_opt:.1f} K")
print(f"  Conversion = {float(info['conversion']['A'])*100:.2f}%")
print(f"\nCosts:")
print(f"  Capital: ${V_opt * 10000:,.0f}")
print(f"  Energy:  ${(T_opt - 300) * 100:,.0f}")
print(f"  Total:   ${V_opt * 10000 + (T_opt - 300) * 100:,.0f}")

## 3. Economic Optimization: Profit Maximization

Now let's maximize **profit** considering revenues and all costs:

$$\text{Profit} = \text{Revenue}(F_B) - \text{Raw Material}(F_A) - \text{Capital}(V) - \text{Energy}(Q)$$

In [ ]:
def profit(params: Array) -> Array:
    """
    Profit = Revenue - Costs
    
    Revenue: $50 per mol/s of B produced
    Costs:
      - Raw material A: $10 per mol/s
      - Capital: $5000 * V per year (annualized)
      - Energy: $0.1 * Q (heat duty in W)
    """
    V, T = params[0], params[1]
    
    cstr = create_cstr(V, {"A": jnp.array(1e6), "Ea": jnp.array(50000.0)})
    inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
    outlet, info = cstr(inlet, T_spec=T)
    
    F_A_in = 10.0
    F_B_out = outlet["F_B"]
    Q = jnp.abs(info["Q"])
    
    # Annual basis (8000 hours/year)
    hours_per_year = 8000.0
    seconds_per_year = hours_per_year * 3600.0
    
    revenue = 50.0 * F_B_out * seconds_per_year / 1e6
    raw_material_cost = 10.0 * F_A_in * seconds_per_year / 1e6
    capital_cost = 5000.0 * V / 1e6
    energy_cost = 0.1 * Q * hours_per_year / 1e6
    
    net_profit = revenue - raw_material_cost - capital_cost - energy_cost
    return -net_profit  # Minimize negative profit


print("Maximizing annual profit...")
print("Revenue: $50/mol B | Costs: A=$10/mol, Capital=$5k/m³, Energy=$0.1/W\n")

x_opt, _ = adam_optimizer(
    profit,
    x0=jnp.array([1.0, 350.0]),
    learning_rate=0.02,
    max_iter=200,
    bounds=(jnp.array([0.1, 300.0]), jnp.array([5.0, 450.0])),
    verbose=False,
)

V_opt, T_opt = float(x_opt[0]), float(x_opt[1])

# Calculate final economics
cstr = create_cstr(jnp.array(V_opt), {"A": jnp.array(1e6), "Ea": jnp.array(50000.0)})
inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
outlet, info = cstr(inlet, T_spec=jnp.array(T_opt))

F_B_out = float(outlet["F_B"])
Q = abs(float(info["Q"]))
hours_per_year = 8000.0
seconds_per_year = hours_per_year * 3600.0

revenue = 50.0 * F_B_out * seconds_per_year / 1e6
raw_material = 10.0 * 10.0 * seconds_per_year / 1e6
capital = 5000.0 * V_opt / 1e6
energy = 0.1 * Q * hours_per_year / 1e6
net_profit = revenue - raw_material - capital - energy

print("✓ Optimal design:")
print(f"  V = {V_opt:.3f} m³, T = {T_opt:.1f} K")
print(f"  Conversion = {float(info['conversion']['A'])*100:.2f}%")
print(f"\nAnnual economics ($M/year):")
print(f"  Revenue (B sales):     ${revenue:6.3f}M")
print(f"  Raw material cost:    -${raw_material:6.3f}M")
print(f"  Capital (annualized): -${capital:6.3f}M")
print(f"  Energy cost:          -${energy:6.3f}M")
print(f"  ─────────────────────────────")
print(f"  Net Profit:            ${net_profit:6.3f}M")

## 4. Parameter Estimation: Fitting to Data

Given experimental conversion data at various temperatures, estimate the kinetic parameters (A, Ea).

This is a classic **inverse problem** that requires minimizing:

$$\text{Loss} = \sum_i (X_{model}(T_i) - X_{exp,i})^2$$

In [ ]:
# Generate synthetic "experimental" data
true_log_A = jnp.log(1e6)
true_Ea = 50000.0

temperatures = jnp.array([320.0, 340.0, 360.0, 380.0, 400.0])

def true_conversion(T):
    cstr = create_cstr(jnp.array(1.0), {"A": jnp.exp(true_log_A), "Ea": true_Ea})
    inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
    _, info = cstr(inlet, T_spec=T)
    return info["conversion"]["A"]

# Add noise to create "experimental" data
key = jax.random.PRNGKey(42)
noise = jax.random.normal(key, shape=(5,)) * 0.02
experimental_X = jnp.array([float(true_conversion(T)) for T in temperatures]) + noise

print("Experimental data (with 2% noise):")
for T, X in zip(temperatures, experimental_X):
    print(f"  T = {float(T):.0f} K: X = {float(X)*100:.2f}%")

In [ ]:
def loss(params: Array) -> Array:
    """Sum of squared errors between model and data."""
    log_A, Ea = params[0], params[1]
    
    def model_X(T):
        cstr = create_cstr(jnp.array(1.0), {"A": jnp.exp(log_A), "Ea": Ea})
        inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
        _, info = cstr(inlet, T_spec=T)
        return info["conversion"]["A"]
    
    predictions = jnp.array([model_X(T) for T in temperatures])
    return jnp.sum((predictions - experimental_X) ** 2)


# Initial guess (deliberately wrong)
x0 = jnp.array([jnp.log(1e5), 40000.0])  # A=1e5, Ea=40 kJ/mol

print(f"Initial guess: A = {jnp.exp(x0[0]):.2e}, Ea = {x0[1]/1000:.1f} kJ/mol")
print(f"Initial loss: {float(loss(x0)):.6f}")

print("\nFitting parameters...")
x_opt, _ = adam_optimizer(
    loss,
    x0,
    learning_rate=0.1,
    max_iter=200,
    bounds=(jnp.array([jnp.log(1e4), 30000.0]), jnp.array([jnp.log(1e8), 70000.0])),
    verbose=False,
)

estimated_A = jnp.exp(x_opt[0])
estimated_Ea = x_opt[1]

print(f"\n✓ Estimated parameters:")
print(f"  A  = {float(estimated_A):.2e} (true: {float(jnp.exp(true_log_A)):.2e})")
print(f"  Ea = {float(estimated_Ea)/1000:.2f} kJ/mol (true: {true_Ea/1000:.2f})")
print(f"\nFinal loss: {float(loss(x_opt)):.8f}")
print(f"\nRelative errors:")
print(f"  A:  {abs(float(estimated_A) - float(jnp.exp(true_log_A)))/float(jnp.exp(true_log_A))*100:.2f}%")
print(f"  Ea: {abs(float(estimated_Ea) - true_Ea)/true_Ea*100:.2f}%")

## 5. Pareto Analysis: Multi-Objective Trade-offs

Often we have competing objectives. The **Pareto front** shows optimal trade-offs.

**Objectives**:
- Maximize conversion
- Minimize cost

We use the **weighted sum method** to generate Pareto-optimal points.

In [ ]:
def cost(V, T):
    return float(V) * 10000.0 + float(T - 300.0) * 100.0

def conversion(V, T):
    cstr = create_cstr(V, {"A": jnp.array(1e6), "Ea": jnp.array(50000.0)})
    inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
    _, info = cstr(inlet, T_spec=T)
    return float(info["conversion"]["A"])


print("Generating Pareto front: Conversion vs Cost\n")

pareto_points = []

for alpha in jnp.linspace(0.01, 0.99, 20):
    def weighted_obj(params):
        V, T = params[0], params[1]
        cstr = create_cstr(V, {"A": jnp.array(1e6), "Ea": jnp.array(50000.0)})
        inlet = make_stream({"A": 10.0, "B": 0.0}, T=300.0, P=101325.0)
        _, info = cstr(inlet, T_spec=T)
        
        X = info["conversion"]["A"]
        C = V * 10.0 + (T - 300.0) * 0.1  # Scaled cost
        return alpha * (-X) + (1 - alpha) * C
    
    x_opt, _ = gradient_descent(
        weighted_obj,
        x0=jnp.array([1.0, 350.0]),
        learning_rate=jnp.array([0.05, 2.0]),
        max_iter=100,
        bounds=(jnp.array([0.1, 300.0]), jnp.array([5.0, 450.0])),
        verbose=False,
    )
    
    X = conversion(x_opt[0], x_opt[1])
    C = cost(x_opt[0], x_opt[1])
    pareto_points.append((X, C, float(x_opt[0]), float(x_opt[1])))

pareto_points.sort(key=lambda p: p[0])

print("Pareto-optimal solutions:")
print("  Conversion   Cost($)    V(m³)   T(K)")
print("  " + "─" * 38)
for X, C, V, T in pareto_points[::4]:  # Show every 4th point
    print(f"    {X*100:5.1f}%    {C:7.0f}   {V:5.2f}   {T:5.1f}")

print("\n📊 Interpretation:")
print("  • Higher conversion requires more capital (V) or energy (T)")
print("  • The Pareto front shows efficient trade-offs")
print("  • Points below the front are sub-optimal")

## Summary

This notebook demonstrated various optimization scenarios:

| Scenario | Method | Key Insight |
|----------|--------|-------------|
| Optimal T | 1D gradient descent | Simple but effective |
| V + T jointly | Adam + penalty | Handle constraints via penalty |
| Economic | Profit maximization | Include all costs |
| Parameter estimation | Fit to data | Inverse problem |
| Pareto analysis | Weighted sum | Multi-objective trade-offs |

**Key advantage of differentiable simulation**: All these optimizations use **exact gradients** from automatic differentiation, making them efficient and reliable.